# 07. Multi-Head Attention и маски

В прошлом блокноте мы реализовали Self-Attention с одной головой. Теперь сделаем следующий шаг: несколько механизмов внимания будут работать параллельно — это **Multi-Head Attention (MHA)**.

Также научимся исключать `<PAD>` из вычисления внимания с помощью padding mask. Это особенно важно, когда предложения имеют разную длину.

## 1. Зачем нужны несколько голов?

Одна голова внимания строит один набор весов внимания. Multi-Head Attention использует несколько таких голов параллельно.

Можно интуитивно представить, что разные головы способны научиться учитывать разные типы связей между токенами. Например, одна голова может уделять внимание близким словам, другая — более удалённым. Но это не жёстко заданные роли: во время обучения модель сама формирует полезные закономерности.

Если размерность модели `d_model = 16`, а голов `n_heads = 4`, то размерность каждой головы:

`head_dim = d_model / n_heads = 4`.

Каждая голова работает с частью общей размерности. После вычисления результаты всех голов объединяются обратно в вектор размерности `d_model`.

## 2. Подготовим batch с разными длинами предложений

Создадим вход из двух предложений длиной по пять позиций. Во втором и первом предложениях часть позиций заполнена `<PAD>`.

Для простоты сразу создадим случайные векторы, как будто они уже прошли через Embedding и Positional Encoding. Маска будет содержать 1 для настоящих токенов и 0 для padding.

In [ ]:
import math
import torch
import torch.nn as nn

torch.manual_seed(42)

batch_size = 2
sequence_length = 5
d_model = 16
n_heads = 4

# Два предложения: у первого 4 настоящих токена, у второго 3.
attention_mask = torch.tensor([
    [1, 1, 1, 1, 0],
    [1, 1, 1, 0, 0]
])

# Случайные входные векторы (B, T, C).
x = torch.randn(batch_size, sequence_length, d_model)

print("Форма x:", x.shape)
print("Маска:\n", attention_mask)
print("Форма маски:", attention_mask.shape)


## 3. Разделяем векторы на головы

Сначала Q, K и V имеют форму `(B, T, d_model)`. Чтобы каждая голова работала отдельно, разделим последнюю размерность на `n_heads` частей.

Например, `(2, 5, 16)` превращается в `(2, 5, 4, 4)`, а затем переставим оси и получим `(2, 4, 5, 4)`:

- 2 — batch;
- 4 — головы;
- 5 — позиции токенов;
- 4 — размерность одной головы.

Попробуй внимательно проследить, как меняется форма тензора.

In [ ]:
head_dim = d_model // n_heads

qkv_example = torch.randn(batch_size, sequence_length, d_model)
split_example = qkv_example.view(batch_size, sequence_length, n_heads, head_dim)
heads_example = split_example.transpose(1, 2)

print("Исходная форма:", qkv_example.shape)
print("После view:", split_example.shape)
print("После transpose:", heads_example.shape)

## 4. Padding mask: не обращаем внимание на `<PAD>`

В предыдущих блокнотах мы добавляли `<PAD>`, чтобы все предложения имели одинаковую длину. Но padding — это не настоящее слово. Модель не должна использовать такие позиции как источник информации.

У нас есть `attention_mask` формы `(B, T)`: 1 — настоящий токен, 0 — padding.

Матрица оценок внимания имеет форму `(B, n_heads, T, T)`. Чтобы запретить всем запросам смотреть на padding-позиции, преобразуем маску к форме `(B, 1, 1, T)`. Затем заменим оценки для padding-ключей на `-inf` перед softmax.

После softmax вес на таких позициях станет нулевым.

In [ ]:
# True там, где позиция является padding.
padding_mask = (attention_mask == 0)

# Добавляем оси, чтобы маска подходила к матрице scores.
padding_mask_for_scores = padding_mask[:, None, None, :]

example_scores = torch.randn(batch_size, n_heads, sequence_length, sequence_length)
masked_scores = example_scores.masked_fill(padding_mask_for_scores, float("-inf"))
example_weights = torch.softmax(masked_scores, dim=-1)

print("Форма padding mask:", padding_mask.shape)
print("Форма маски для scores:", padding_mask_for_scores.shape)
print("Форма scores:", example_scores.shape)
print("Веса на padding-позициях для первого примера:\n", example_weights[0, 0, :, 4])
print("Суммы весов по строкам:\n", example_weights.sum(dim=-1))

Мы маскируем **позиции Key**, то есть столбцы, на которые могут распределяться веса внимания. Позиции Query, соответствующие padding, всё ещё могут получить некоторый выходной вектор. Для классификации мы позже не будем учитывать padding при объединении представлений предложения.

Отдельно существует **causal mask** — она запрещает токену смотреть на будущие позиции. Она нужна для авторегрессионной генерации текста (например, GPT). Для нашей первой задачи классификации тональности мы используем двунаправленное внимание и causal mask не применяем.

## 5. Реализуем Multi-Head Attention как модуль

Теперь соединим всё в один класс. В нём будут четыре обучаемые линейные проекции:

- `q_proj`, `k_proj`, `v_proj` — создают Q, K и V;
- `out_proj` — преобразует объединённый результат всех голов.

Параметр `n_heads` должен делить `d_model` без остатка. Иначе размерность нельзя будет равномерно разделить между головами.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError("d_model должен делиться на n_heads без остатка")

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, attention_mask=None):
        # x: (B, T, C)
        batch_size, seq_len, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # Разделяем общую размерность на головы: (B, H, T, head_dim).
        q = q.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        # Считаем scaled dot-product attention.
        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        # Исключаем padding-позиции из набора Key.
        if attention_mask is not None:
            padding_mask = (attention_mask == 0)[:, None, None, :]
            scores = scores.masked_fill(padding_mask, float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        context = weights @ v

        # Объединяем головы обратно в (B, T, C).
        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, seq_len, self.d_model)

        output = self.out_proj(context)
        return output, weights


mha = MultiHeadSelfAttention(d_model=d_model, n_heads=n_heads)
output, weights = mha(x, attention_mask)

print("Форма входа:", x.shape)
print("Форма выхода:", output.shape)
print("Форма attention weights:", weights.shape)
print("Количество параметров:", sum(p.numel() for p in mha.parameters()))

## 6. Сравним головы внимания

Посмотрим на веса разных голов для первого предложения. У каждой головы своя матрица внимания. Значения могут отличаться, потому что каждая голова использует свою часть проекций Q, K и V.

In [ ]:
print("Голова 1, веса внимания первого предложения:")
print(weights[0, 0])

print("\nГолова 2, веса внимания первого предложения:")
print(weights[0, 1])

print("\nПроверка суммы весов (для каждого токена и каждой головы):")
print(weights.sum(dim=-1))

print("\nВеса на padding-позиции последнего столбца первого примера:")
print(weights[0, :, :, 4])

## 7. Меняется ли число параметров при увеличении количества голов?

Это важный момент для наших будущих экспериментов. При фиксированном `d_model` и обычных полных линейных проекциях число параметров MHA не зависит от количества голов. Меняется то, как общая размерность разделяется между головами.

Проверим это экспериментально.

In [ ]:
for heads in [1, 2, 4, 8, 16]:
    layer = MultiHeadSelfAttention(d_model=16, n_heads=heads)
    number_of_parameters = sum(p.numel() for p in layer.parameters())
    print(f"Голов: {heads:2d} | Размерность одной головы: {16 // heads:2d} | Параметров: {number_of_parameters}")

## 8. Самостоятельные упражнения

1. Поменяй `n_heads` с 4 на 2. Как изменится `head_dim` и форма `weights`?
2. Поменяй `n_heads` на 8. Сравни веса внимания разных голов. Одинаковые ли они?
3. Временно вызови модель без `attention_mask`: `mha(x)`. Посмотри на веса последнего столбца — будут ли там ненулевые значения?
4. Попробуй создать слой с `d_model=15` и `n_heads=4`. Объясни, почему возникнет ошибка.
5. Почему при фиксированном `d_model` увеличение `n_heads` само по себе не увеличивает число параметров этих четырёх линейных слоёв?
6. Объясни разницу между padding mask и causal mask.

Не забывай: если меняешь `d_model` или `n_heads`, они должны быть согласованы и делиться без остатка.

## Что мы узнали

Теперь у нас есть Multi-Head Self-Attention с padding mask. Мы умеем:

1. Проецировать входы в Q, K и V.
2. Разделять размерность между несколькими головами.
3. Рассчитывать внимание отдельно для каждой головы.
4. Исключать padding-позиции из внимания.
5. Объединять результаты голов обратно в исходную размерность.

**Следующий этап — Transformer Block.** Добавим Feed-Forward Network, LayerNorm и Residual Connections. После него подготовительная серия закончится: начнём собирать основную модель проекта для классификации тональности.